# **라이브러리 설치 및 도구 임포트**

In [ ]:
!pip install PyPDF2 datasets sentence-transformers==3.4.1

In [ ]:
import os
import requests
import json
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm
from openai import OpenAI
from torch.utils.data import DataLoader
from sentence_transformers import SentenceTransformer, losses, InputExample
from sentence_transformers.evaluation import InformationRetrievalEvaluator
import torch
from sklearn.metrics.pairwise import cosine_similarity
import PyPDF2

In [ ]:
# .env 파일에서 환경 변수 로드
load_dotenv("/content/env")
# 환경 변수에서 API 키 가져오기
api_key = os.getenv("OPENAI_API_KEY")

# **PDF 추출**

In [ ]:
# PDF 파일 다운로드
urls = [
    "https:/raw.githubusercontent.com/langchain-kr/langchain-tutorial/main/ch09.%20Embedding%20Fine-tuning/ict_japan_2024.pdf",
    "https:/rwa.githubusercontent.com/langchain-kr/langchain-tutorial/main/ch09.%20Embedding%20Fine-tuning/ict_usa_2024.pfd"
]

for url in urls:
  filename = url.split("/")[-1]
  response = requests.get(url)

  with open(filename, "wb") as f:
    f.write(response.content)

  print(f"{filanem} 다운로드 완료")

In [ ]:
def extract_text_from_pdf(pdf_path):
  # PDF 파일에서 텍스트를 추출하는 함수
  text_chunks = []

  with open(pdf_path, 'rb') as file:
    pdf_reader = PyPDF2.PdfReader(file)

    for page_num in range(len(pdf_reader.pages)):
      page = pdf_reader.pages[page_num]
      text = page.extract_text()

      # 페이지 단위로 청크 생성
      if text.strip():
        text = text.strip()

        # 문서 길이가 10자 초과인 경우만 추가
        if len(text) > 10:
          text_chunks.append(text)

  return text_chunks

# 미국 ICT 동향(학습 데이터)
train_corpus = extract_text_from_pdf('ict_usa_2024.pdf')
print(f'학습 데이터 문서 개수: {len(train_corpus)}')

# 일본 ICT 동량(검증 데이터)
val_corpus = extract_text_from_pdf('ict_japan_2024.pdf')
print(f'검증 데이터 문서 개수: {len(val_corpus)}')

In [ ]:
print('10번 문서:', train_corpus[10])